In [1]:
import numpy as np

In [5]:
def entropy(y):
    p = np.mean(y)
    if(p == 0 or p == 1): 
        return 0.0
    return -1 * p * np.log2(p) -1 * (1-p) * np.log2(1-p)

In [6]:
print(entropy(np.array([1, 1, 0, 0])))  
print(entropy(np.array([1, 1, 1, 1])))  
print(entropy(np.array([1, 1, 1, 0]))) 

1.0
0.0
0.8112781244591328


In [9]:
def information_gain(y_parent, y_left, y_right):
    return entropy(y_parent) - ((y_left.size/y_parent.size) * entropy(y_left) + (y_right.size/y_parent.size) * entropy(y_right))

In [10]:
parent = np.array([1, 1, 0, 0])
left = np.array([1, 1])
right = np.array([0, 0])

print(information_gain(parent, left, right))

1.0


In [14]:
x = [2, 4, 7, 10]
def create_thresholds(x):
    x = np.sort(np.unique(x))
    y = []
    for i in range(len(x)-1):
        y.append((x[i] + x[i+1])/2)
    return y
create_thresholds(x)

[np.float64(3.0), np.float64(5.5), np.float64(8.5)]

In [17]:
def split(x, y, threshold):
    y_left = []
    y_right = []
    for i in range(len(y)):
        if(x[i] <= threshold):
            y_left.append(y[i])
        else:
            y_right.append(y[i])
    return np.array(y_left), np.array(y_right)

In [18]:
x = np.array([18, 25, 40, 52])
y = np.array([0, 0, 1, 1])

print(split(x, y, 30))

(array([0, 0]), array([1, 1]))


In [19]:
def best_split_for_feature(x, y):
    thresholds = create_thresholds(x)
    best_threshold = -1
    best_gain = -1
    for i in range(len(thresholds)):
        y_left, y_right = split(x,y,thresholds[i])
        if(information_gain(y,y_left,y_right) > best_gain):
            best_gain = information_gain(y,y_left,y_right)
            best_threshold = thresholds[i]
    return best_threshold, best_gain

In [20]:
x = np.array([2, 4, 7, 10])
y = np.array([0, 0, 1, 1])

print(best_split_for_feature(x, y))

(np.float64(5.5), np.float64(1.0))


In [21]:
X = np.array([
    [2, 10],
    [4, 20],
    [7, 15],
    [10, 25]
])

y = np.array([0, 0, 1, 1])

In [23]:
def best_split(X, y):
    best_feature = None
    best_threshold = None
    best_gain = -np.inf

    for j in range(X.shape[1]):
        threshold,gain = best_split_for_feature(X[:,j],y)
        if(gain > best_gain):
            best_feature = j
            best_threshold = threshold
            best_gain = gain
        
    return best_feature, best_threshold, best_gain

In [25]:
def split_data(X, y, feature, threshold):
    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right = X[right_mask]
    y_right = y[right_mask]

    return X_left, y_left, X_right, y_right

In [26]:
def build_tree(X, y, depth=0, max_depth=3, min_samples_split=2):
    if(np.all(y == y[0])):
        return {"leaf": True, "prediction" : y[0]}
    if(depth >= max_depth or len(y) < min_samples_split):
        return {"leaf": True, "prediction" : int(np.mean(y) >= 0.5)}

    feature, threshold, gain = best_split(X, y)
    if gain <= 0:
        return {"leaf": True, "prediction": int(np.mean(y) >= 0.5)}

    X_left, y_left, X_right, y_right = split_data(X, y, feature, threshold)
    left_tree = build_tree(X_left,y_left,depth + 1,max_depth,min_samples_split)
    right_tree = build_tree(X_right,y_right,depth + 1,max_depth,min_samples_split)

    return {
        "leaf": False,
        "feature": feature,
        "threshold": threshold,
        "left": left_tree,
        "right": right_tree
    }

In [30]:
def predict_one(x, tree):
    if(tree["leaf"] == True):
        return tree["prediction"]
    
    if(x[tree["feature"]] > tree["threshold"]):
        return predict_one(x,tree["right"])
    else:
        return predict_one(x,tree["left"])

In [31]:
def predict(X, tree):
    predictions = []

    for row in X:
        predictions.append(predict_one(row,tree))

    return np.array(predictions)

In [32]:
X = np.array([
    [2, 10],
    [4, 20],
    [7, 15],
    [10, 25]
])

y = np.array([0, 0, 1, 1])

tree = build_tree(X, y, max_depth=3)

print(tree)
print(predict(X, tree))
print(y)

{'leaf': False, 'feature': 0, 'threshold': np.float64(5.5), 'left': {'leaf': True, 'prediction': np.int64(0)}, 'right': {'leaf': True, 'prediction': np.int64(1)}}
[0 0 1 1]
[0 0 1 1]


In [33]:
X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [6]
])

y = np.array([0, 0, 1, 1, 0, 0])

tree = build_tree(X, y, max_depth=3)

print(tree)
print(predict(X, tree))
print(y)

{'leaf': False, 'feature': 0, 'threshold': np.float64(2.5), 'left': {'leaf': True, 'prediction': np.int64(0)}, 'right': {'leaf': False, 'feature': 0, 'threshold': np.float64(4.5), 'left': {'leaf': True, 'prediction': np.int64(1)}, 'right': {'leaf': True, 'prediction': np.int64(0)}}}
[0 0 1 1 0 0]
[0 0 1 1 0 0]
